### import essential libraries

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import tensorflow as tf

### Load Data

In [2]:
(X, y), (_, _) = tf.keras.datasets.cifar10.load_data()

170498071/170498071 [==============================] - 6s 0us/step


### turning the problem to binary classsification

In [3]:
y = np.where(np.isin(y, [0, 1, 2, 3, 4]), 0, 1).reshape(-1)

### Preprocessing

In [4]:
# Standardize the data
scaler = StandardScaler()
X = X.reshape(X.shape[0], -1)
X = scaler.fit_transform(X)

### train- test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define imbalance ratios

In [6]:
imbalance_ratios = [(1, 99), (5, 95), (10, 90), (20, 80), (30, 70)]

### Define a function to imbalance the data

In [7]:
def create_imbalanced_dataset(X, y, ratio):
    unique_classes = np.unique(y)
    if len(unique_classes) < 2:
        raise ValueError('The dataset must contain at least two classes.')
    
    X_imbalanced, y_imbalanced = [], []
    for i, cls in enumerate(unique_classes):
        samples = X[y == cls]
        n_samples = min(max(int(len(X) * ratio[i] / (ratio[0] + ratio[1])), 1), len(samples))
        
        resampled_samples = resample(samples, replace=False, n_samples=n_samples, random_state=42)
        
        X_imbalanced.append(resampled_samples)
        y_imbalanced.extend([cls] * n_samples)
    
    X_imbalanced = np.vstack(X_imbalanced)
    y_imbalanced = np.array(y_imbalanced)
    
    return X_imbalanced, y_imbalanced

### for each ratio implement three ideas and evaluate them

In [8]:
for ratio in imbalance_ratios:
    
    # Create imbalanced training set
    X_train_imbalanced, y_train_imbalanced = create_imbalanced_dataset(X_train, y_train, ratio)


    # Train and evaluate logistic regression on imbalanced dataset
    clf_imbalanced = LogisticRegression(max_iter=1000)
    clf_imbalanced.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_imbalanced = clf_imbalanced.predict(X_test)
    

    # Apply RandomOverSampler to balance the dataset by oversampling the minority class
    random_over_sampler = RandomOverSampler(sampling_strategy='minority', random_state=42)
    X_train_balanced_with_ros, y_train_balanced_with_ros = random_over_sampler.fit_resample(X_train_imbalanced, y_train_imbalanced)

    # Train and evaluate logistic regression on balanced dataset (with randomm over sampling)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_balanced_with_ros, y_train_balanced_with_ros)
    y_pred_balanced_with_ros = model.predict(X_test)


    # Create and train the logistic regression model with class weighting
    model_with_class_weighting = LogisticRegression(class_weight='balanced', random_state=42)
    model_with_class_weighting.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_class_weighting = model_with_class_weighting.predict(X_test)

    

    # Train and evaluate XGBClassifier with logistic regression as the base learner
    xgb_clf = XGBClassifier(booster='gblinear', objective='binary:logistic', n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_clf.fit(X_train_imbalanced, y_train_imbalanced)
    y_pred_balanced_with_xgb = xgb_clf.predict(X_test)


    # Calculate performance metrics
    metrics_imbalanced = [accuracy_score(y_test, y_pred_imbalanced), precision_score(y_test, y_pred_imbalanced), recall_score(y_test, y_pred_imbalanced), f1_score(y_test, y_pred_imbalanced)]
    metrics_balanced_with_ros = [accuracy_score(y_test, y_pred_balanced_with_ros), precision_score(y_test, y_pred_balanced_with_ros), recall_score(y_test, y_pred_balanced_with_ros), f1_score(y_test, y_pred_balanced_with_ros)]
    metrics_balanced_with_class_weighting = [accuracy_score(y_test, y_pred_balanced_with_class_weighting), precision_score(y_test, y_pred_balanced_with_class_weighting), recall_score(y_test, y_pred_balanced_with_class_weighting), f1_score(y_test, y_pred_balanced_with_class_weighting)]
    metrics_balanced_with_xgb= [accuracy_score(y_test, y_pred_balanced_with_xgb), precision_score(y_test, y_pred_balanced_with_xgb), recall_score(y_test, y_pred_balanced_with_xgb), f1_score(y_test, y_pred_balanced_with_xgb)]

    print(f"Imbalance ratio: {ratio[0]}:{ratio[1]}")
    print("Imbalanced dataset metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_imbalanced))
    print("Balanced dataset with random over sampling metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_ros))
    print("Balanced dataset with class weighting metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_class_weighting))
    print("Balanced dataset with xgb metrics: Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1-score: {:.4f}".format(*metrics_balanced_with_xgb))
    print("\n")

/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

Imbalance ratio: 1:99
Imbalanced dataset metrics: Accuracy: 0.5093, Precision: 0.5079, Recall: 0.9872, F1-score: 0.6707
Balanced dataset with random over sampling metrics: Accuracy: 0.5244, Precision: 0.5174, Recall: 0.8985, F1-score: 0.6567
Balanced dataset with class weighting metrics: Accuracy: 0.5283, Precision: 0.5212, Recall: 0.8392, F1-score: 0.6430
Balanced dataset with xgb metrics: Accuracy: 0.5066, Precision: 0.5064, Recall: 0.9996, F1-score: 0.6722




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

Imbalance ratio: 5:95
Imbalanced dataset metrics: Accuracy: 0.5135, Precision: 0.5101, Recall: 0.9795, F1-score: 0.6709
Balanced dataset with random over sampling metrics: Accuracy: 0.5503, Precision: 0.5425, Recall: 0.7130, F1-score: 0.6161
Balanced dataset with class weighting metrics: Accuracy: 0.5685, Precision: 0.5612, Recall: 0.6770, F1-score: 0.6137
Balanced dataset with xgb metrics: Accuracy: 0.5079, Precision: 0.5071, Recall: 0.9994, F1-score: 0.6728




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

Imbalance ratio: 10:90
Imbalanced dataset metrics: Accuracy: 0.5193, Precision: 0.5135, Recall: 0.9575, F1-score: 0.6685
Balanced dataset with random over sampling metrics: Accuracy: 0.5590, Precision: 0.5543, Recall: 0.6578, F1-score: 0.6016
Balanced dataset with class weighting metrics: Accuracy: 0.5753, Precision: 0.5731, Recall: 0.6308, F1-score: 0.6006
Balanced dataset with xgb metrics: Accuracy: 0.5121, Precision: 0.5093, Recall: 0.9943, F1-score: 0.6735




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

Imbalance ratio: 20:80
Imbalanced dataset metrics: Accuracy: 0.5448, Precision: 0.5298, Recall: 0.8949, F1-score: 0.6656
Balanced dataset with random over sampling metrics: Accuracy: 0.5818, Precision: 0.5809, Recall: 0.6241, F1-score: 0.6017
Balanced dataset with class weighting metrics: Accuracy: 0.5966, Precision: 0.6005, Recall: 0.6069, F1-score: 0.6037
Balanced dataset with xgb metrics: Accuracy: 0.5328, Precision: 0.5212, Recall: 0.9459, F1-score: 0.6721




/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.9/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

Imbalance ratio: 30:70
Imbalanced dataset metrics: Accuracy: 0.5674, Precision: 0.5503, Recall: 0.7951, F1-score: 0.6505
Balanced dataset with random over sampling metrics: Accuracy: 0.5818, Precision: 0.5853, Recall: 0.5964, F1-score: 0.5908
Balanced dataset with class weighting metrics: Accuracy: 0.5970, Precision: 0.6035, Recall: 0.5944, F1-score: 0.5989
Balanced dataset with xgb metrics: Accuracy: 0.5727, Precision: 0.5500, Recall: 0.8570, F1-score: 0.6700


